# CIFAKE — Real vs. Deepfake Detection (ViT-B/16)

End-to-end pipeline: download → preprocess → **two-stage fine-tuning** → full metrics → **OOD benchmark on CIFAKE** → ONNX export.

**Before you run:**
1. Set the runtime to **GPU**: *Runtime → Change runtime type → Hardware accelerator → GPU (T4)*.
2. Add your Kaggle token via **Colab Secrets** (the 🔑 icon in the left sidebar) as `KAGGLE_API_TOKEN`. **Do not paste tokens into cells.**

> Approx. runtime on a T4: a few minutes to download, then ~20–30 min/epoch over 100k images.

## 1. Setup & reproducibility

In [ ]:
# Install dependencies
!pip install -q kaggle kagglesdk onnx onnxscript scikit-learn

import os, random, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("[OK] Imports ready. Torch:", torch.__version__)

## 2. Kaggle credentials (secure — no hard-coded tokens)

In [ ]:
# Store your token in Colab Secrets (the key icon in the left sidebar) as KAGGLE_API_TOKEN.
# This keeps secrets OUT of the notebook so they are never committed or shared.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    print("[OK] Kaggle token loaded from Colab Secrets.")
except Exception as e:
    print("[!] Could not load token from Colab Secrets:", e)
    print("    Option A: click the key icon (Secrets) and add KAGGLE_API_TOKEN, then re-run this cell.")
    print("    Option B: upload kaggle.json (Kaggle -> Settings -> API -> Create New Token):")
    # Uncomment the two lines below to upload kaggle.json instead:
    # from google.colab import files; files.upload()
    # !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 3. Download datasets

In [ ]:
# 140k Real and Fake Faces  -> train / val / test
# CIFAKE                    -> out-of-distribution (OOD) benchmark
print("[SYSTEM] Downloading 140k Faces (~3.75 GB, a few minutes)...")
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p /content/data/140k_faces

print("[SYSTEM] Downloading CIFAKE benchmark...")
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images --unzip -p /content/data/cifake

# Locate the 140k base dir (the one that contains train/valid/test)
path_140k = None
for root, dirs, _ in os.walk('/content/data/140k_faces'):
    if {'train', 'valid', 'test'}.issubset(set(dirs)):
        path_140k = root; break
path_cifake = '/content/data/cifake'
assert path_140k, "[ERROR] 140k train/valid/test folders not found."
print(f"[OK] 140k at:   {path_140k}")
print(f"[OK] CIFAKE at: {path_cifake}")

## 4. Device (GPU required)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[SYSTEM] Device: {device}")
if device.type != "cuda":
    print("\n*** WARNING: No GPU detected. Training a ViT on 100k images on CPU is impractical.")
    print("    Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4), then re-run. ***")
else:
    print("[OK] GPU:", torch.cuda.get_device_name(0))

USE_AMP = device.type == "cuda"   # mixed precision only makes sense on GPU

## 5. Data: transforms, datasets, loaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Augmentation on TRAIN improves robustness; eval/test stay clean.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ---- Toggle: include CIFAKE's TRAIN split in training? --------------------------------
#  True  -> the model also learns CIFAKE; its test set becomes IN-DISTRIBUTION (high acc).
#  False -> face-only training; CIFAKE stays a held-out OOD benchmark (the original setup).
INCLUDE_CIFAKE_IN_TRAIN = True
# ---------------------------------------------------------------------------------------

from torch.utils.data import ConcatDataset, Subset

# 140k faces
train_dataset = datasets.ImageFolder(f"{path_140k}/train", transform=train_transform)
val_dataset   = datasets.ImageFolder(f"{path_140k}/valid", transform=eval_transform)
test_dataset  = datasets.ImageFolder(f"{path_140k}/test",  transform=eval_transform)

# CIFAKE test (used as the CIFAKE benchmark in both modes)
cifake_dataset = datasets.ImageFolder(f"{path_cifake}/test", transform=eval_transform)

if INCLUDE_CIFAKE_IN_TRAIN:
    # Two views of CIFAKE train: augmented (for training) and clean (for validation).
    cif_train_aug  = datasets.ImageFolder(f"{path_cifake}/train", transform=train_transform)
    cif_train_eval = datasets.ImageFolder(f"{path_cifake}/train", transform=eval_transform)
    # Carve a disjoint 5% validation slice from CIFAKE train (fixed seed, no leakage).
    n = len(cif_train_aug)
    perm = np.random.default_rng(SEED).permutation(n)
    n_val = int(0.05 * n)
    cif_val_idx, cif_tr_idx = perm[:n_val], perm[n_val:]
    cifake_train_ds = Subset(cif_train_aug,  cif_tr_idx.tolist())
    cifake_val_ds   = Subset(cif_train_eval, cif_val_idx.tolist())
    combined_train = ConcatDataset([train_dataset, cifake_train_ds])
    combined_val   = ConcatDataset([val_dataset,   cifake_val_ds])
    print("[MIX] Training on 140k faces + CIFAKE train -> CIFAKE test is now IN-DISTRIBUTION.")
else:
    combined_train, combined_val = train_dataset, val_dataset
    print("[OOD] Face-only training -> CIFAKE test stays an out-of-distribution benchmark.")

BATCH, NUM_WORKERS = 64, 2
def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

train_loader  = make_loader(combined_train, True)
val_loader    = make_loader(combined_val,   False)
test_loader   = make_loader(test_dataset,   False)   # 140k faces test (always in-distribution)
cifake_loader = make_loader(cifake_dataset, False)   # CIFAKE test

print(f"Train: {len(combined_train)} | Val: {len(combined_val)} | "
      f"140k Test: {len(test_dataset)} | CIFAKE Test: {len(cifake_dataset)}")
print("140k classes:", train_dataset.class_to_idx, "| CIFAKE classes:", cifake_dataset.class_to_idx)
# Both map fake -> 0, real -> 1, so labels line up across datasets.

## 6. Model: ViT-B/16 (ImageNet pretrained)

In [ ]:
def build_model():
    m = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    for p in m.parameters():                 # start fully frozen (Stage 1 = linear probe)
        p.requires_grad = False
    in_features = m.heads.head.in_features
    m.heads.head = nn.Linear(in_features, 2) # new head, trainable by default
    return m.to(device)

model = build_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"[OK] ViT-B/16 ready. Trainable (head only): {trainable:,} / {total:,}")

## 7. Metrics + training helpers

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Returns accuracy, precision, recall, F1, ROC-AUC and a confusion matrix."""
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            out = model(images)
            if criterion is not None:
                total_loss += criterion(out, labels).item()
        prob_real = torch.softmax(out.float(), dim=1)[:, 1]  # P(class 'real')
        y_true.append(labels.cpu()); y_pred.append(out.argmax(1).cpu()); y_prob.append(prob_real.cpu())
    y_true = torch.cat(y_true).numpy(); y_pred = torch.cat(y_pred).numpy(); y_prob = torch.cat(y_prob).numpy()
    return {
        "loss": (total_loss / len(loader)) if criterion is not None else None,
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }

def print_metrics(name, m):
    print(f"\n=== {name} ===")
    if m['loss'] is not None: print(f"Loss     : {m['loss']:.4f}")
    print(f"Accuracy : {m['accuracy']*100:.2f}%")
    print(f"Precision: {m['precision']*100:.2f}%   Recall: {m['recall']*100:.2f}%   F1: {m['f1']*100:.2f}%")
    print(f"ROC-AUC  : {m['roc_auc']:.4f}")
    print("Confusion matrix [rows = true (fake,real), cols = pred (fake,real)]:")
    print(m['confusion_matrix'])

def train_stage(model, epochs, lr, stage_name, patience=2, ckpt="best_model.pth"):
    """Train currently-unfrozen params with AMP, an LR scheduler, early stopping,
    and best-checkpoint saving (by validation accuracy)."""
    params    = [p for p in model.parameters() if p.requires_grad]
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
    scaler    = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    best_acc, no_improve = 0.0, 0
    for epoch in range(epochs):
        t0 = time.time(); model.train()
        run_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                out = model(images); loss = criterion(out, labels)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            run_loss += loss.item()
            correct  += (out.argmax(1) == labels).sum().item(); total += labels.size(0)
        val = evaluate(model, val_loader, criterion)
        scheduler.step(val['accuracy'])
        print(f"[{stage_name}] Epoch {epoch+1}/{epochs} ({time.time()-t0:.0f}s) "
              f"train_loss={run_loss/len(train_loader):.4f} train_acc={100*correct/total:.2f}% "
              f"val_acc={val['accuracy']*100:.2f}% val_f1={val['f1']*100:.2f}%")
        if val['accuracy'] > best_acc:
            best_acc, no_improve = val['accuracy'], 0
            torch.save(model.state_dict(), ckpt)
            print(f"   -> new best (val_acc={best_acc*100:.2f}%) saved to {ckpt}")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"   -> early stop (no val improvement for {patience} epochs)"); break
    return best_acc

## 8. Stage 1 — linear probe (train the head only)

In [ ]:
print("[SYSTEM] Stage 1: training the classification head with the backbone frozen...")
best1 = train_stage(model, epochs=2, lr=1e-3, stage_name="probe", ckpt="best_model.pth")
print(f"[OK] Stage 1 best val accuracy: {best1*100:.2f}%")

## 9. Stage 2 — fine-tune the last transformer blocks

In [ ]:
# Restore the best head, then unfreeze the last 4 encoder blocks + final norm and
# fine-tune everything that is now trainable at a LOW learning rate.
model.load_state_dict(torch.load("best_model.pth", map_location=device))
for p in model.encoder.layers[-4:].parameters():
    p.requires_grad = True
for p in model.encoder.ln.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[SYSTEM] Stage 2: fine-tuning. Trainable params now: {trainable:,}")
best2 = train_stage(model, epochs=3, lr=1e-5, stage_name="finetune", ckpt="best_model.pth")
print(f"[OK] Stage 2 best val accuracy: {best2*100:.2f}%")

## 10. Final in-distribution evaluation (140k test set)

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
test_metrics = evaluate(model, test_loader, nn.CrossEntropyLoss())
print_metrics("140k Test (in-distribution)", test_metrics)

## 11. CIFAKE evaluation

With `INCLUDE_CIFAKE_IN_TRAIN = True` (cell under *5. Data*) the model has seen CIFAKE's **train**
split, so this **CIFAKE test set is now in-distribution** and accuracy should be high. Set the toggle
to `False` to restore the original **out-of-distribution** generalization benchmark.

In [ ]:
cifake_metrics = evaluate(model, cifake_loader, nn.CrossEntropyLoss())
mode = "in-distribution (trained on CIFAKE train)" if INCLUDE_CIFAKE_IN_TRAIN else "out-of-distribution (held out)"
print_metrics(f"CIFAKE -- {mode}", cifake_metrics)
if not INCLUDE_CIFAKE_IN_TRAIN:
    print("\nNote: CIFAKE is a different domain (32x32 objects, Stable Diffusion) than the 140k face")
    print("training data (256px faces, StyleGAN). A large accuracy drop here is EXPECTED and measures")
    print("cross-generator / cross-domain generalization.")

## 12. Export to ONNX

In [ ]:
model.eval()
dummy = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "deepfake_vit_detector.onnx"
torch.onnx.export(
    model, dummy, onnx_path,
    export_params=True, opset_version=14, do_constant_folding=True,
    input_names=['input_image'], output_names=['prediction'],
    dynamic_axes={'input_image': {0: 'batch'}, 'prediction': {0: 'batch'}},
)
print(f"[OK] Exported model to {onnx_path}")

## 13. (Optional) Save artifacts to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy('best_model.pth',               '/content/drive/MyDrive/best_model.pth')
# shutil.copy('deepfake_vit_detector.onnx',   '/content/drive/MyDrive/deepfake_vit_detector.onnx')